# Embedding-Based Deduplication

Use embedding deduplication inside a `QuestionPipeline` to remove near-duplicate generated questions before labeling. This is useful when several seed articles describe the same event, or when the question generator produces semantically equivalent phrasings that exact matching would miss.

In this notebook, we build a small forecasting pipeline and add `EmbeddingDeduplication` between question generation and labeling.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Build a forecasting pipeline

This pipeline uses news articles about AI policy and chips. These topics often create duplicate questions because multiple articles can cover the same export rule, company announcement, or regulatory decision.

In [3]:
from datetime import datetime
from lightningrod import (
    BinaryAnswerType,
    ForwardLookingQuestionGenerator,
    NewsSeedGenerator,
    QuestionPipeline,
    WebSearchLabeler,
)

answer_type = BinaryAnswerType()

seed_generator = NewsSeedGenerator(
    start_date=datetime(2025, 9, 1),
    end_date=datetime(2025, 10, 1),
    interval_duration_days=7,
    articles_per_search=10,
    search_query="AI chip export controls semiconductor regulation",
)

question_generator = ForwardLookingQuestionGenerator(
    instructions=(
        "Generate forward-looking questions about AI chip export controls, semiconductor supply chains, "
        "and government regulation. Questions should be resolvable by public reporting."
    ),
    answer_type=answer_type,
    questions_per_seed=3,
)

labeler = WebSearchLabeler(
    answer_type=answer_type,
    confidence_threshold=0.5,
)

## Add embedding deduplication

`EmbeddingDeduplication` compares semantic similarity, not just exact strings. The most important knobs are:

- `fields`: sample fields concatenated before embedding. Start with `question_text`; add `resolution_criteria` when duplicate meaning depends on how the question resolves.
- `key_fields`: exact-match buckets. Here, `date_close` prevents questions with different close dates from removing each other.
- `similarity_threshold`: lower values remove more aggressively; higher values keep more near-duplicates.
- `synonyms`: domain-specific regex replacements applied before embedding. Keep these conservative because bad aliases can merge distinct questions.

In [ ]:
from lightningrod import EmbeddingDeduplication, EmbeddingDeduplicationSynonyms

deduplication = EmbeddingDeduplication(
    fields=["question_text", "resolution_criteria"],
    key_fields=["date_close"],
    similarity_threshold=0.91,
    synonyms=EmbeddingDeduplicationSynonyms.from_dict({
        r"\bAI\b": "artificial intelligence",
        r"\bU\.S\.\b|\bUS\b": "united states",
        r"\bsemiconductors?\b": "chip",
    }),
    normalize_numbers=True,
)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    deduplication=deduplication,
    labeler=labeler,
)

## Estimate cost and run a small test

Start small and inspect the pipeline summary. The deduplication step should appear between question generation and labeling, with its input and output counts showing how many generated questions were removed before labeling.

In [5]:
estimated_cost = lr.transforms.estimate_cost(pipeline, max_seeds=5)
print(f"Estimated cost: ${estimated_cost:.2f}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> API Error: estimate cost                                                                                    │
│                                                                                                                 │
│  Failed to estimate cost: [ValidationError(loc=['body', 'config', 'QUESTION_PIPELINE', 'deduplication',         │
│  'config_type'], msg="Input should be <TransformType.FUZZY_DEDUPLICATION: 'FUZZY_DEDUPLICATION'>",              │
│  type_='literal_error', input_='EMBEDDING_DEDUPLICATION',                                                       │
│  ctx=ValidationErrorContext(additional_properties={'expected': "<TransformType.FUZZY_DEDUPLICATION:             │
│  'FUZZY_DEDUPLICATION'>"}), additional_properties={})]                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Exception: Failed to estimate cost: [ValidationError(loc=['body', 'config', 'QUESTION_PIPELINE', 'deduplication', 'config_type'], msg="Input should be <TransformType.FUZZY_DEDUPLICATION: 'FUZZY_DEDUPLICATION'>", type_='literal_error', input_='EMBEDDING_DEDUPLICATION', ctx=ValidationErrorContext(additional_properties={'expected': "<TransformType.FUZZY_DEDUPLICATION: 'FUZZY_DEDUPLICATION'>"}), additional_properties={})]

In [ ]:
dataset = lr.transforms.run(
    pipeline,
    max_seeds=5,
    max_cost_dollars=1.00,
    name="embedding-dedup-example",
)

print(f"Created dataset {dataset.id} with {dataset.num_rows} rows")

## Inspect the retained samples

Review a few rows to confirm the remaining questions are meaningfully distinct. If you still see duplicate phrasing, lower `similarity_threshold` slightly or add conservative synonyms. If unrelated questions are being merged, raise the threshold or add another exact bucket such as `event_date`.

In [ ]:
from pprint import pprint
from lightningrod.display import flatten_samples

rows = flatten_samples(dataset.samples())

for row in rows[:5]:
    pprint({
        "question_text": row.get("question_text"),
        "date_close": row.get("date_close"),
        "label": row.get("label"),
        "is_valid": row.get("is_valid"),
    })
    print("---")

## Tuning checklist

- Use `key_fields` for facts that should never be merged across, especially dates in forecasting datasets.
- Lower `similarity_threshold` only after checking false positives; values around `0.90` to `0.93` are a practical starting range.
- Add `synonyms` for stable domain aliases like tickers, agency abbreviations, or common acronyms.
- Turn on `normalize_numbers=True` when questions mention quantities, prices, vote counts, revenue, market caps, or deadlines.